In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic"
os.environ['MKL_THREADING_LAYER'] = "GNU"

In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [5]:
if is_main:
    if is_jupyter: 
        # Basics 
        seed        = 42
        environment_string = "cart_pole"
        gold_timesteps =10
        training_timesteps = 10
        num_concepts_selected = 3
        selection_function = "q_value"
        # Experiment #1 & #2
        run_basic = False
        run_iterative = False 
        run_two_stage = True   
        run_imperfect=False 
        run_intervention=False 
        # Experiment #3
        cbm_accuracy_by_concept = None 
        intervention_probability = 0
        intervention_accuracy_by_concept = None 
        cbm_std_by_concept = None 
        target_abstraction = 0.05
        reward_error = 0
        # Experiment #4
        concept_source = "human_selected_binary"
        # Experiment #5
        assess_completeness=False
        # Experiment #6
        num_iterations = 0
        selections_per_round = 0
        initial_concepts = 0
        out_folder = "llm"
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument('--seed', help='Random Seed', type=int, default=42)
        parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
        parser.add_argument('--training_timesteps', help='Number of training timesteps', type=int, default=10000)
        parser.add_argument('--gold_timesteps', help='Number of training timesteps without concepts', type=int, default=10000)
        parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
        parser.add_argument('--selection_function', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
        parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--intervention_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--cbm_std_by_concept', help="What is the error of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--run_two_stage', help='Run the two stage?', action='store_true')
        parser.add_argument('--run_iterative', help='Run the iterative?', action='store_true')
        parser.add_argument('--run_intervention', help='Run the intervention?', action='store_true')
        parser.add_argument('--run_basic', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_imperfect', help='Run the imperfect comparisons?', action='store_true')
        parser.add_argument('--intervention_probability', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
        parser.add_argument('--concept_source', help='When selecting, use q_value, policy, or transition?', type=str, default="human_selected")
        parser.add_argument('--assess_completeness', help='Compare to the concept completeness algorithm?', action='store_true')
        parser.add_argument('--num_iterations', help='Number of iterations for iterative algorithms',type=int, default=0)
        parser.add_argument('--selections_per_round', help='Concepts to select per round',type=int, default=0)
        parser.add_argument('--initial_concepts', help='Number of starting/initial concepts',type=int, default=0)
        parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

        args = parser.parse_args()

        seed = args.seed
        environment_string = args.environment_string
        training_timesteps = args.training_timesteps 
        gold_timesteps = args.gold_timesteps
        num_concepts_selected = args.num_concepts_selected
        selection_function = args.selection_function
        cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
        cbm_std_by_concept = args.cbm_std_by_concept
        run_basic = args.run_basic
        run_iterative = args.run_iterative
        run_two_stage = args.run_two_stage
        run_imperfect = args.run_imperfect
        run_intervention = args.run_intervention
        intervention_probability = args.intervention_probability
        intervention_accuracy_by_concept = args.intervention_accuracy_by_concept
        target_abstraction = args.target_abstraction
        reward_error = args.reward_error
        concept_source = args.concept_source
        assess_completeness = args.assess_completeness
        num_iterations = args.num_iterations 
        selections_per_round = args.selections_per_round
        initial_concepts = args.initial_concepts
        out_folder = args.out_folder

    save_name = secrets.token_hex(4)  

In [6]:
if is_main:
        results = {}
        results['parameters'] = {'seed'      : seed,
                'environment_string'    : environment_string, 
                'training_timesteps': training_timesteps, 
                'gold_timesteps': gold_timesteps,
                'selection_function': selection_function,
                'num_concepts_selected': num_concepts_selected,
                'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
                'cbm_std_by_concept': cbm_std_by_concept,
                'intervention_probability': intervention_probability,
                'intervention_accuracy_by_concept': intervention_accuracy_by_concept,
                'target_abstraction': target_abstraction,
                'reward_error': reward_error, 
                'concept_source': concept_source,
                'assess_completeness': assess_completeness,
                'num_iterations': num_iterations,
                'selections_per_round': selections_per_round, 
                'initial_concepts': initial_concepts,
                'run_basic': run_basic,
                'run_iterative': run_iterative, 
                'run_two_stage': run_two_stage, 
                'run_intervention': run_intervention,
                'run_imperfect': run_imperfect, 
        }
        print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'cart_pole', 'training_timesteps': 10, 'gold_timesteps': 10, 'selection_function': 'q_value', 'num_concepts_selected': 3, 'cbm_accuracy_by_concept': None, 'cbm_std_by_concept': None, 'intervention_probability': 0, 'intervention_accuracy_by_concept': None, 'target_abstraction': 0.05, 'reward_error': 0, 'concept_source': 'human_selected_binary', 'assess_completeness': False, 'num_iterations': 0, 'selections_per_round': 0, 'initial_concepts': 0, 'run_basic': False, 'run_iterative': False, 'run_two_stage': True, 'run_intervention': False, 'run_imperfect': False}


In [7]:
if is_main:
    np.random.seed(seed)
    random.seed(seed)

In [9]:
vec_env, gym_env, additional_info = get_environment("glucose",None,seed)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

## Glucose

In [128]:
register(
    id="simglucose/adolescent2-custom-v0",
    entry_point=GlucoseEnvironment,  # adjust if using a module
    max_episode_steps=288,
    kwargs={"patient_name": "adolescent#002"},
)

env = gym.make("simglucose/adolescent2-custom-v0", render_mode=None)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment simglucose/adolescent2-custom-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [129]:
env.reset()

(array([ 0.79087526,  0.        ,  0.        ,  0.        , -0.25881904,
         0.9659258 ], dtype=float32),
 {'sample_time': 3.0,
  'patient_name': 'adolescent#002',
  'meal': 0,
  'patient_state': array([  0.        ,   0.        ,   0.        , 282.34854415,
          33.12995603,   6.01930508,   0.        , 119.18      ,
         119.18      ,   4.96870842,  77.84847603,  47.17212488,
         290.83646231]),
  'time': datetime.datetime(2018, 1, 1, 23, 0),
  'bg': 158.1750488437134,
  'lbgi': 0.0,
  'hbgi': 4.0527862649498,
  'risk': 4.0527862649498})

In [130]:

for i in range(10):
    obs, info = env.reset()
    done = False
    steps = 0
    total_reward = 0
    while not done:
        obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
        done = terminated or truncated
        steps += 1
        total_reward += reward 

    print("Steps in first episode:", steps,total_reward)


Steps in first episode: 34 -5.68895954812809
Steps in first episode: 34 -4.668417387970048
Steps in first episode: 35 -6.048828515747974
Steps in first episode: 34 -5.128570888405775
Steps in first episode: 33 -4.928351832094445
Steps in first episode: 35 -5.849172162105564
Steps in first episode: 37 -4.949209080618912
Steps in first episode: 34 -5.6284547393180375
Steps in first episode: 35 -4.74926453799627
Steps in first episode: 34 -5.728626119199126


In [131]:
num_envs = 8




/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [132]:
envs.reset()

array([[ 0.7582959 ,  0.        ,  0.        ,  0.        , -0.9659258 ,
         0.25881904],
       [ 0.7697481 ,  0.        ,  0.        ,  0.        , -0.8660254 ,
         0.5       ],
       [ 0.75251037,  0.        ,  0.        ,  0.        ,  0.8660254 ,
         0.5       ],
       [ 0.74845684,  0.        ,  0.        ,  0.        , -0.25881904,
        -0.9659258 ],
       [ 0.8001694 ,  0.        ,  0.        ,  0.        , -0.9659258 ,
        -0.25881904],
       [ 0.770856  ,  0.        ,  0.        ,  0.        ,  0.8660254 ,
        -0.5       ],
       [ 0.76711106,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.7867681 ,  0.        ,  0.        ,  0.        ,  0.70710677,
         0.70710677]], dtype=float32)

In [119]:
wandb.finish()


In [120]:


name = "glucose_ppo_breakthrough"
wandb.init(
    project="Concept Decisions",
    name=name,
    config=model_params
)

model = PPO(
    policy="MlpPolicy",
    env=envs,
    verbose=0,
    learning_rate=model_params["learning_rate"],
    n_steps=model_params["n_steps"],
    batch_size=model_params["batch_size"],
    n_epochs=model_params["n_epochs"],
    gamma=model_params["gamma"],
    gae_lambda=model_params["gae_lambda"],
    clip_range=model_params["clip_range"],
    ent_coef=model_params["ent_coef"],
    vf_coef=model_params["vf_coef"],
    max_grad_norm=model_params["max_grad_norm"],
    tensorboard_log=f"./runs/{name}",
    device='cpu'
)

# Train for longer to see breakthrough
total_timesteps = 2_000_000  # 2.5x more training

model.learn(
    total_timesteps=total_timesteps,
    callback=WandbLoggingCallback(),
    progress_bar=True
)

wandb.finish()


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

In [ ]:
name = "glucose_ppo_breakthrough"
wandb.init(
    project="Concept Decisions",
    name=name,
    config=model_params
)

model.learn(
    total_timesteps=total_timesteps,
    callback=WandbLoggingCallback(),
    progress_bar=True
)


In [ ]:
wandb.finish()

approx_kl,███████████▆▆▆▆▆▆▆▆▆▆▆▂▂▂▂▂▂▂▂▂████████▁
clip_fraction,███████▆▆▆▆▆▆▆▆▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
ema_norm_reward,█▇▇▄▄▅▄▅▂▃▄▁▂▄▄▅▂▃▄▅▃▂▂▅▂▃▄▄▃▄▅▂▃▄▂▃▃▄▄▄
entropy_loss,▁▁▁▁▁▁▁▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆█████
episode_length_mean,▁▃▂▅▂▁▃▂▂▁▂▂▂▃▆▂▆▇▃▂▂▆▅▆▂▁▂▂▁▃▆▂█▂▂▂▆▃▄▃
episode_reward_max,▇▆▇█▇▇▇█▇▇▇██▇▇▇▇██▇██▇▇▁█▇█▇█▇▇███▇▇▇█▃
episode_reward_mean,▆██▇▇▁▇██▃▇▂▅█▇█▄▇▇███▅▇▅▇▇█▆██▃█▃██████
episode_reward_min,▇▂▇▇▇▇▇▇▂▇▆▇▁▂▇▁▃▇█▇▇▁▂▇▇▇▁█▇▇▁█▇█▇▇█▇▁█
episodes_completed,▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇████
explained_variance,███████████▇▇▇▇▇▇▇▇▁▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▇
+1,...
